# Garbage Detection with YOLOv5
### Computer Vision Portfolio WS2026, Assignment 1

**The scenario.** A recycling plant wants to sort waste automatically. A camera looks at the items on a conveyor
belt; a model must find every item and say what material it is, so a robot arm can pick it up and drop it in the
right bin. That is an **object detection** task: for every object we need a **box** (where) and a **class** (what).

**This notebook tells the whole story, from raw data to an explained result**, and gives the reason for every
choice on the way:

| Part | Question it answers | Assignment task |
|---|---|---|
| 1. Setup | Where is everything? | |
| 2. The dataset | What exactly did we get? | Task 1 |
| 3. Dataset analysis | What makes this data easy or hard? | Task 1 |
| 4. Pipeline decisions | What do we do about it, and why? | Task 1 |
| 5. Data preparation | Clean and split the data | Task 1 |
| 6. Training | How do we train YOLOv5, and why these settings? | Task 2 |
| 7. Test-set evaluation | How good is the model, per category? | Task 2 |
| 8. Error analysis | Which errors happen, and how often? | Task 3 |
| 9. Example detections | What do correct, missed and wrong detections look like? | Task 3 |
| 10. Limitations and improvements | Why do the errors happen, and what would help? | Task 2 + 3 |

The data preparation and training logic lives in the project's `Code/` folder (`config.py`, `data.py`,
`train.py`), so it can also run from the command line. This notebook **imports** those modules instead of copying
them, and adds the analysis and evaluation around them.

---
## 1. Setup

The notebook expects this project layout:

```
project/
├── garbage_detection.ipynb     (this notebook)
├── Code/config.py, data.py, train.py
├── GarbageDataset/images/, labels/      the unzipped dataset
└── weights/best.pt                      the trained model (created by training, or already in the repo)
```

**On Google Colab**, the next cell mounts Google Drive and moves into the project folder. Change `DRIVE_PROJECT` if
your folder has another name. On your own computer the cell does nothing.

Training needs a GPU: on Colab choose *Runtime → Change runtime type → T4 GPU*. The analysis parts run on any CPU.

In [ ]:
import os

DRIVE_PROJECT = "/content/drive/MyDrive/Computer_Vision"   # project folder in Google Drive

try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir(DRIVE_PROJECT)
    print("Running on Colab, project folder:", os.getcwd())
except ImportError:
    print("Running locally, project folder:", os.getcwd())

Install the only dependency, `ultralytics` (it brings PyTorch, NumPy, OpenCV, pandas and matplotlib). Skip this
cell if it is already installed.

In [ ]:
%pip install -q ultralytics

Now the imports. `Code/` is added to the Python path so we can import our own modules: `config` holds every
setting, `data` the preparation functions, `train` the training function.

In [ ]:
import sys
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image as ShowImage, display
from PIL import Image, ImageDraw, ImageFont

sys.path.insert(0, str(Path("Code").resolve()))
import config
import data as prep          # Code/data.py
import train as trainer     # Code/train.py

from config import CLASS_NAMES, DATASET_DIR, RESULTS_DIR, WEIGHTS_DIR

# One consistent look for all charts
BLUE, ORANGE, AQUA, YELLOW, RED = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e34948"
TEXT, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
CLASS_COLORS = [BLUE, ORANGE, AQUA, YELLOW, "#e87ba4", "#008300"]
plt.rcParams.update({
    "font.size": 11, "axes.edgecolor": GRID, "axes.labelcolor": MUTED, "xtick.color": MUTED,
    "ytick.color": MUTED, "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110,
})

print("Dataset folder:", DATASET_DIR, "| exists:", (DATASET_DIR / "images").is_dir())

---
## 2. The dataset

The dataset is a Roboflow export in **YOLO format**: an `images/` folder and a `labels/` folder, matched by file
name. Each label file has one line per object:

```
class  x_center  y_center  width  height        e.g.   2  0.50  0.40  0.30  0.60
```

All four box values are **fractions of the image size** (0 to 1), measured from the top-left corner. The box
extends half its width to the left and right of the center, and half its height up and down.

Three things are **not** included and we have to work out ourselves:
* **Class names.** The IDs follow alphabetical order (0 biodegradable … 5 plastic); the filename prefixes confirm it
  (every `glass…` file is mostly class 2, and so on).
* **A train/validation/test split.** We create one in part 5.
* **A `data.yaml`** that tells YOLO where the data is. `data.py` writes it.

We load everything with `prep.load_dataset()` from `Code/data.py`. It reads every label line, keeps the valid boxes,
and reports lines that are broken (we look at those in part 3.5).

In [ ]:
records, invalid = prep.load_dataset(DATASET_DIR)

n_boxes = sum(len(r["boxes"]) for r in records)
print(f"{len(records):,} images, {n_boxes:,} valid boxes, {sum(invalid.values())} invalid boxes")

sizes = Counter(Image.open(r["image"]).size for r in records[::50])   # every 50th image is enough
print("Image sizes (sample):", dict(sizes))
print("Example record:", {k: (v if k != "boxes" else v[:2]) for k, v in records[0].items()})

Every image is **416×416**, and every image has a label file. Each record holds the image path, the label path,
the image class taken from the file name (`source`) and the list of valid boxes.

---
## 3. Dataset analysis

Before choosing anything about training, we look at the data. We ask four questions, because each one changes how
we should train and evaluate:

1. Are the classes balanced?
2. How many objects are in one image?
3. How big are the objects?
4. Do classes appear together, and are the labels clean?

### 3.1 Are the classes balanced?
A detector learns from **boxes**, not from images, so we count both.

In [ ]:
images_per_class = Counter(r["source"] for r in records)
boxes_per_class = Counter(b[0] for r in records for b in r["boxes"])

balance = pd.DataFrame({
    "images": [images_per_class[n] for n in CLASS_NAMES],
    "boxes": [boxes_per_class[i] for i in range(len(CLASS_NAMES))],
}, index=CLASS_NAMES)
balance["share of boxes %"] = (100 * balance["boxes"] / balance["boxes"].sum()).round(1)
balance["boxes per image"] = (balance["boxes"] / balance["images"]).round(1)
display(balance)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, title in [(axes[0], "images", "Images per class"), (axes[1], "boxes", "Boxes (objects) per class")]:
    bars = ax.bar(CLASS_NAMES, balance[col], color=BLUE, width=0.6)
    ax.bar_label(bars, labels=[f"{v:,}" for v in balance[col]], padding=3, fontsize=9, color=TEXT)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=25)
    ax.yaxis.grid(True, color=GRID)
    ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

print(f"Images: largest/smallest class = {balance['images'].max() / balance['images'].min():.1f} : 1")
print(f"Boxes:  largest/smallest class = {balance['boxes'].max() / balance['boxes'].min():.1f} : 1")

**What we see.** The **images** are roughly balanced (about 2.3 : 1 between the largest and smallest class), but
the **boxes** are not: biodegradable holds about **61% of all boxes**, about 10 times as many as paper. Its images
show piles of fruit and vegetables with many items each.

**Why it matters.** The detector will see far more biodegradable examples than anything else. An overall score can
look fine while a rare class is poor.

**Decision.** Report results **per class**, not only as one overall number.

### 3.2 How many objects are in one image?

In [ ]:
per_image = pd.Series([len(r["boxes"]) for r in records])
print(f"Boxes per image: median {per_image.median():.0f}, mean {per_image.mean():.1f}, max {per_image.max()}")
print(f"Images with exactly 1 box: {100 * (per_image == 1).mean():.1f}%   with 10 or more: {100 * (per_image >= 10).mean():.1f}%")

fig, ax = plt.subplots(figsize=(10, 4))
ax.boxplot([[len(r["boxes"]) for r in records if r["source"] == n] for n in CLASS_NAMES],
           tick_labels=CLASS_NAMES, widths=0.5,
           boxprops=dict(color=BLUE), medianprops=dict(color=TEXT, linewidth=2),
           whiskerprops=dict(color=BLUE), capprops=dict(color=BLUE),
           flierprops=dict(marker="o", markersize=3, markerfacecolor=BLUE, markeredgecolor="none", alpha=0.3))
ax.set_yscale("log")
ax.set_ylabel("boxes per image (log scale)")
ax.set_title("Objects per image, by image class")
ax.yaxis.grid(True, color=GRID)
plt.tight_layout()
plt.show()

**What we see.** Half the images contain a single object, but some contain **hundreds** (the maximum is 358), and
almost all of those are biodegradable piles.

**Why it matters.** YOLO keeps at most 300 detections per image by default. In the densest images that could cut
off correct detections and lower recall.

**Decision.** Allow up to **500 detections per image** in evaluation.

### 3.3 How big are the objects?
We measure a box's size as the square root of its area, in percent of the image side (10% = a box about 42×42 px
in a 416×416 image). A box covering less than 1% of the image counts as **small**.

In [ ]:
SMALL_AREA = 0.01   # < 1% of the image area = smaller than about 42x42 px at 416x416

sizes_rows = []
for i, name in enumerate(CLASS_NAMES):
    areas = np.array([b[3] * b[4] for r in records for b in r["boxes"] if b[0] == i])
    sizes_rows.append({"class": name, "median area %": round(100 * np.median(areas), 1),
                       "small boxes %": round(100 * (areas < SMALL_AREA).mean(), 1)})
display(pd.DataFrame(sizes_rows).set_index("class"))

fig, ax = plt.subplots(figsize=(10, 4))
ax.boxplot([[100 * np.sqrt(b[3] * b[4]) for r in records for b in r["boxes"] if b[0] == i]
            for i in range(len(CLASS_NAMES))],
           tick_labels=CLASS_NAMES, widths=0.5, showfliers=False,
           boxprops=dict(color=BLUE), medianprops=dict(color=TEXT, linewidth=2),
           whiskerprops=dict(color=BLUE), capprops=dict(color=BLUE))
ax.axhline(100 * np.sqrt(SMALL_AREA), color=MUTED, linestyle="--", linewidth=1)
ax.text(6.45, 100 * np.sqrt(SMALL_AREA) + 1.5, "small (< 1% of image)", ha="right", color=MUTED, fontsize=9)
ax.set_ylabel("box size: sqrt(w·h), % of image side")
ax.set_title("Box size per class")
ax.yaxis.grid(True, color=GRID)
plt.tight_layout()
plt.show()

**What we see.** Object size depends strongly on the class. About **half of the biodegradable boxes are small**;
paper and cardboard are large (median area around 12–20% of the image).

**Why it matters.** Small objects cover only a few pixels after the network's downsampling, so they are the hardest
to find. **Prediction: biodegradable will have the lowest recall**, paper and cardboard should be easy to locate.

**Decision.** Train at **640 px** instead of the native 416 px: upscaling gives small objects more pixels in the
network's feature maps. YOLOv5's multi-scale head (part 6) also helps.

### 3.4 Do classes appear together?
Each image was collected for one class (its file name), but it can contain boxes of other classes.

In [ ]:
cooc = np.zeros((len(CLASS_NAMES), len(CLASS_NAMES)), dtype=int)   # rows: image class, columns: box class
for r in records:
    for b in r["boxes"]:
        cooc[CLASS_NAMES.index(r["source"]), b[0]] += 1
share = cooc / cooc.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(7, 5.6))
im = ax.imshow(share, cmap="Blues", vmin=0, vmax=100)
ax.set_xticks(range(6), CLASS_NAMES, rotation=30, ha="right")
ax.set_yticks(range(6), CLASS_NAMES)
ax.set_xlabel("class of the labelled box")
ax.set_ylabel("image class (file name)")
for i in range(6):
    for j in range(6):
        if cooc[i, j]:
            ax.text(j, i, f"{share[i, j]:.0f}%\n({cooc[i, j]:,})", ha="center", va="center", fontsize=8,
                    color="white" if share[i, j] > 55 else TEXT)
ax.set_title("Which boxes appear in which images")
fig.colorbar(im, ax=ax, fraction=0.046, label="% of boxes in that image class")
plt.tight_layout()
plt.show()

mixed = np.mean([len({b[0] for b in r["boxes"]}) > 1 for r in records])
print(f"Images with more than one class: {100 * mixed:.1f}%")

**What we see.** About 90% of images contain one class only. **Glass images are the exception**: about a quarter of
their boxes are metal or plastic, because **lids and caps** on jars and bottles are labelled separately.

**Why it matters.** The model must tell apart objects that touch each other and belong to one physical item.
**Prediction: glass / metal / plastic confusions.**

**Decision.** Nothing to change in the data, but we check this specifically in the error analysis (part 8).

### 3.5 Are the labels clean?
`load_dataset()` rejected every label line that is broken: not 5 values, an unknown class, a value outside 0–1, or a
box with **zero width or height**.

In [ ]:
print(f"Images with invalid label lines: {len(invalid)}, invalid lines in total: {sum(invalid.values())}")
for name, n in list(invalid.items())[:20]:
    print(f"  {name.split('_jp')[0]:<22} {n} invalid line(s)")
if not invalid:
    print("  None found: the label files were already cleaned by an earlier run of data.py.")

**What we found in the raw data:** **17 boxes with a width or height of 0**, out of 74,073. A box with no area cannot
contain an object, and its IoU (overlap) with any prediction is undefined. In every case the real object next to it
already has a correct box, so these are accidental clicks during annotation.

**Decision.** **Remove them** before training. Nothing is lost, and they would otherwise count as "missed objects" in
the evaluation. (If you run this notebook after `data.py` has cleaned the labels, the list above is empty.)

**Other observations from looking at sample images** (next cell): a mix of studio photos on white backgrounds and
real scenes, some stock-photo watermarks, some mirrored copies, and piles that are boxed item by item in some
images but as one group in others. The last point will matter for the error analysis.

In [ ]:
rng = np.random.default_rng(11)
font = ImageFont.load_default()
fig, axes = plt.subplots(2, 6, figsize=(18, 6.4))
for col, name in enumerate(CLASS_NAMES):
    pool = [r for r in records if r["source"] == name]
    for row, r in enumerate(rng.choice(pool, 2, replace=False)):
        im = Image.open(r["image"]).convert("RGB")
        W, H = im.size
        d = ImageDraw.Draw(im)
        for c, x, y, w, h in r["boxes"]:
            d.rectangle([(x - w / 2) * W, (y - h / 2) * H, (x + w / 2) * W, (y + h / 2) * H],
                        outline=CLASS_COLORS[c], width=2)
        axes[row, col].imshow(im)
        axes[row, col].axis("off")
        axes[row, col].set_title(f"{name} ({len(r['boxes'])} boxes)" if row == 0 else f"{len(r['boxes'])} boxes",
                                 fontsize=10, loc="center")
plt.suptitle("Sample images with their labelled boxes", x=0.01, ha="left", fontweight="bold")
plt.tight_layout()
plt.show()

---
## 4. Pipeline decisions

Every training choice follows from a finding above:

| Finding | Decision | Reason |
|---|---|---|
| No split provided | **70 / 15 / 15** train / val / test, **stratified by class**, seed 42 | Enough data to learn (≈7,300 images) and a test set big enough for stable per-class results (≈175 plastic images). Stratified, so every split has the same class mix. Fixed seed, so the split is reproducible. |
| The lecture exercise used 90/10 train/val | Add a separate **test** set | Validation is used for early stopping and picking the best epoch, so it is no longer "unseen". The test set is used **once**, at the end. |
| 17 zero-size boxes | **Remove** them | No area, no IoU; they are annotation mistakes. |
| Many boxes under 42×42 px | Train at **640 px** | More pixels for small objects. |
| Biodegradable = 61% of boxes | Report **per-class** AP | So rare classes are not hidden by the dominant one. |
| Up to 358 objects per image | Up to **500 detections** per image in evaluation | Default 300 would cap recall in dense piles. |
| Varied backgrounds, mirrored copies already present | **Default YOLO augmentation**: mosaic, horizontal flip, HSV colour, scale, translate; **no vertical flip** | More variety from 7,300 images. Waste on a belt has a natural up/down orientation, so vertical flips would create unrealistic images. |
| Only about 10k images | **Transfer learning** from COCO-pretrained weights | The backbone already knows general shapes and textures; we only adapt it. |
| Risk of overfitting | **Early stopping** (patience 15) on validation mAP | Stop when the model stops improving on unseen data. |

**Why YOLOv5** (and not another detector):
* It is the detector the lecture focuses on.
* It is a **one-stage** detector: one forward pass predicts all boxes, about 10 ms per image on a T4 GPU. That is
  real-time, which a conveyor belt needs. Two-stage detectors like Faster R-CNN can be slightly better on tiny
  objects but are several times slower.
* The labels are already in YOLO format, and `ultralytics` provides training, augmentation and mAP out of the box.

---
## 5. Data preparation

`prep.main()` from `Code/data.py` does three things:
1. removes the invalid boxes from the label files,
2. splits the images 70/15/15 per class with seed 42,
3. writes `train.txt`, `val.txt`, `test.txt` (lists of image paths) and `data.yaml` into `GarbageDataset/`.

**Why lists instead of copying images into folders?** YOLO accepts both. Lists are a *virtual* split: no image is
duplicated, which is much faster (especially on Google Drive) and saves 200 MB. YOLO finds each label by replacing
`/images/` with `/labels/` in the image path.

In [ ]:
prep.main()
print()
print(config.DATA_YAML.read_text())

Check that the split is really stratified: every class should be split 70 / 15 / 15.

In [ ]:
def read_split(name):
    return {Path(line).name for line in (DATASET_DIR / f"{name}.txt").read_text().split()}

split_of = {}
for name in ("train", "val", "test"):
    for img in read_split(name):
        split_of[img] = name

check = pd.crosstab(pd.Series([prep.image_class(n) for n in split_of], name="class"),
                    pd.Series(list(split_of.values()), name="split"))[["train", "val", "test"]]
check["val %"] = (100 * check["val"] / check.sum(axis=1)).round(1)
check["test %"] = (100 * check["test"] / check[["train", "val", "test"]].sum(axis=1)).round(1)
display(check)

---
## 6. Training

### The model: YOLOv5s
YOLOv5 has three parts (as in the lecture):
* **Backbone** (CSP bottleneck blocks, SiLU activation): extracts features from the image.
* **Neck** (SPPF + PANet): combines features from several scales, so both large and small objects are represented.
* **Head**: predicts boxes and class scores on **three grid scales** (80×80, 40×40, 20×20 at 640 px). Overlapping
  predictions are filtered with **Non-Maximum Suppression (NMS)**.

We use the **small** version (`yolov5s`, 9.1 M parameters): big enough for 6 classes, small enough to train in
about 2 hours on a free Colab T4. *Note:* the `ultralytics` package ships YOLOv5 as "YOLOv5u", with the same
backbone and neck but an anchor-free head instead of the 9 anchor boxes shown in the lecture.

### The settings (all in `Code/config.py`)
| Setting | Value | Reason |
|---|---|---|
| Pretrained weights | `yolov5su.pt` (COCO) | Transfer learning; only the 6-class head is new |
| Image size | 640 | Small objects (part 3.3) |
| Epochs | 50 | Enough to converge when fine-tuning |
| Early stopping | patience 15 | Stop if validation mAP stops improving |
| Batch size | 16 | Fits a T4's 15 GB at 640 px |
| Optimizer / learning rate | chosen by ultralytics (AdamW, lr 0.001) | Standard for this dataset size |
| Augmentation | ultralytics defaults (part 4) | Mosaic is switched off for the last 10 epochs, so training ends on realistic images |
| Seed | 42 | Reproducible |

### Run
`trainer.train()` from `Code/train.py` trains and copies the best checkpoint (by validation mAP) to
`weights/best.pt`. **Training takes about 2 hours on a T4**, so the cell skips it when the weights already exist.
Set `RETRAIN = True` to train again.

In [ ]:
RETRAIN = False

best_weights = WEIGHTS_DIR / "best.pt"
if best_weights.exists() and not RETRAIN:
    print(f"{best_weights} already exists: skipping training (set RETRAIN = True to train again).")
else:
    best_weights = trainer.train()
    print("Best model saved to", best_weights)

### Training curves
`ultralytics` logs every epoch to `results.csv`: the three training losses (box position, class, distribution
focal loss) and the validation precision, recall and mAP.

In [ ]:
log_file = RESULTS_DIR / "runs" / config.RUN_NAME / "results.csv"
if log_file.exists():
    log = pd.read_csv(log_file)
    log.columns = log.columns.str.strip()
    col = lambda key: next(c for c in log.columns if key in c)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    axes[0].plot(log["epoch"], log[col("mAP50(B)")], color=BLUE, lw=2, label="mAP@0.5")
    axes[0].plot(log["epoch"], log[col("mAP50-95(B)")], color=ORANGE, lw=2, label="mAP@[.50:.95]")
    axes[0].set_title("Validation mAP per epoch")
    axes[0].set_xlabel("epoch")
    axes[0].legend(frameon=False)
    for key, c, label in [("train/box_loss", BLUE, "box"), ("train/cls_loss", ORANGE, "class"), ("train/dfl_loss", AQUA, "DFL")]:
        axes[1].plot(log["epoch"], log[col(key)], color=c, lw=2, label=label)
    axes[1].set_title("Training losses per epoch")
    axes[1].set_xlabel("epoch")
    axes[1].legend(frameon=False)
    for ax in axes:
        ax.yaxis.grid(True, color=GRID)
    plt.tight_layout()
    plt.show()
    best_epoch = log[col("mAP50-95(B)")].idxmax()
    print(f"Best validation epoch: {int(log.loc[best_epoch, 'epoch'])}, "
          f"mAP@0.5 {log.loc[best_epoch, col('mAP50(B)')]:.3f}, mAP@[.50:.95] {log.loc[best_epoch, col('mAP50-95(B)')]:.3f}")
else:
    print(f"No training log at {log_file} (it exists only where the model was trained).")

**How to read it (our run).** Validation mAP rises steadily and is **still improving slightly at epoch 50**, so early
stopping never triggered: more epochs could help a little. The losses fall continuously and validation does not
get worse, so there is **no sign of overfitting**. The early epochs are noisy while the new 6-class head adapts.
Best validation result: mAP@0.5 = 0.676, mAP@[.50:.95] = 0.474.

---
## 7. Test-set evaluation

The test set (1,571 images, 11,031 objects) was never used for training or for choosing the best epoch, so this is
an honest estimate of how the model performs on new images.

**The metrics** (lecture recap):
* A prediction is **correct** if its box overlaps a ground-truth box with **IoU ≥ 0.5** and the class matches.
  IoU (intersection over union) = overlap area / combined area.
* **Precision** = share of predictions that are correct. **Recall** = share of real objects that were found.
* **AP** (average precision) = area under the precision–recall curve of one class; **mAP** = mean over the 6 classes.
* **mAP@0.5** uses IoU ≥ 0.5. **mAP@[.50:.95]** averages over IoU thresholds 0.50, 0.55 … 0.95 and therefore also
  rewards **precise** boxes (the COCO standard).
* **F1** = 2·P·R / (P + R), the balance of precision and recall at one confidence threshold.

*Why no accuracy?* Accuracy needs true negatives ("correctly no object here"), and in detection every empty box
position would be one: millions per image. Detection therefore uses precision, recall, F1 and mAP instead.

`Code/evaluate.py` runs the evaluation; the same function is used here, so the notebook and the command line
(`python Code/evaluate.py`) always report the same numbers.

In [ ]:
from ultralytics import YOLO

import evaluate as evaluator          # Code/evaluate.py
from config import EVAL_DIR, MAX_DET  # MAX_DET = 500: some images contain more than 300 objects (part 3.2)

overall, rows = evaluator.evaluate()  # model.val(...) on the test split, see Code/evaluate.py
evaluator.save_results(overall, rows)
evaluator.print_results(overall, rows)

per_class = (pd.DataFrame(rows).set_index("class")
             .rename(columns={"f1": "F1", "AP50": "AP@0.5", "AP50-95": "AP@[.50:.95]"}).round(3))
display(per_class)

model = YOLO(str(best_weights))       # loaded again for the predictions in part 8

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
y = np.arange(len(per_class))
ax.barh(y - 0.18, per_class["AP@0.5"], 0.36, color=BLUE, label="AP@0.5")
ax.barh(y + 0.18, per_class["AP@[.50:.95]"], 0.36, color=ORANGE, label="AP@[.50:.95]")
for yy, v in zip(y, per_class["AP@0.5"]):
    ax.text(v + 0.01, yy - 0.18, f"{v:.2f}", va="center", fontsize=9)
for yy, v in zip(y, per_class["AP@[.50:.95]"]):
    ax.text(v + 0.01, yy + 0.18, f"{v:.2f}", va="center", fontsize=9)
ax.set_yticks(y, per_class.index)
ax.set_xlim(0, 0.9)
ax.set_title("Test AP per class")
ax.legend(frameon=False, loc="lower right")
ax.xaxis.grid(True, color=GRID)
plt.tight_layout()
plt.show()

for f in ["confusion_matrix_normalized.png", "PR_curve.png", "BoxPR_curve.png"]:
    if (EVAL_DIR / "test" / f).exists():
        display(ShowImage(str(EVAL_DIR / "test" / f), width=650))

**Interpretation (our run: test mAP@0.5 = 0.645, mAP@[.50:.95] = 0.444, precision 0.70, recall 0.57).**

* **Generalization.** Test mAP is close to validation mAP (0.676 / 0.474), so the model works on new images and did
  not overfit.
* **Glass is the easiest class** (AP@[.50:.95] ≈ 0.57, recall ≈ 0.72): large, distinctive bottles and jars, and the
  most training images.
* **Biodegradable has the lowest AP@[.50:.95]** (≈ 0.34) although its AP@0.5 is average (≈ 0.61): the model finds
  the items roughly, but placing exact boxes on tiny, crowded objects is hard. This confirms the prediction from
  part 3.3.
* **Plastic has the lowest AP@0.5** (≈ 0.60): the fewest training images, and it often looks like metal or glass.
* **Paper and cardboard** are large, yet only mid-table: they share colour and texture.
* **Precision is higher than recall**: when the model detects something it is usually right; its main weakness is
  objects it **does not find**. Part 8 measures this directly.

---
## 8. Error analysis

mAP summarises everything in one number. To understand **which** errors happen, we match every prediction to the
ground truth ourselves and sort each box into one of four outcomes:

| Outcome | Meaning |
|---|---|
| **correct** | prediction overlaps a ground-truth box (IoU ≥ 0.5) and the class is right |
| **wrong class** | prediction overlaps a ground-truth box (IoU ≥ 0.5) but the class is wrong |
| **false positive** | prediction overlaps no ground-truth box (something detected where nothing is labelled) |
| **missed** | ground-truth box that no prediction overlaps (false negative) |

**Matching rule.** Predictions with confidence ≥ 0.25 (the usual operating point) are processed from the most to the
least confident. Each one takes the best-overlapping, still unmatched ground-truth box, preferring a box of the same
class. Every ground-truth box can be matched only once, so a duplicate detection counts as a false positive.

In [ ]:
CONF_THRESHOLD = 0.25   # operating point: keep predictions at least this confident
IOU_MATCH = 0.5         # same IoU threshold as mAP@0.5
OUTCOMES = {"correct": AQUA, "wrong_class": YELLOW, "false_pos": RED, "missed": BLUE}


def load_gt(label_path, W, H):
    # YOLO label file -> (N, 4) pixel boxes [x1, y1, x2, y2] and (N,) class IDs
    boxes, classes = [], []
    for line in Path(label_path).read_text().splitlines():
        p = line.split()
        if len(p) == 5:
            c, x, y, w, h = int(p[0]), *map(float, p[1:])
            boxes.append([(x - w / 2) * W, (y - h / 2) * H, (x + w / 2) * W, (y + h / 2) * H])
            classes.append(c)
    return np.array(boxes, dtype=float).reshape(-1, 4), np.array(classes, dtype=int)


def iou_matrix(a, b):
    # IoU between every box in a (N, 4) and every box in b (M, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)


def match_image(pred_boxes, pred_cls, pred_conf, gt_boxes, gt_cls, iou_thr=IOU_MATCH):
    # Greedy matching, most confident prediction first. Returns one outcome dict per box.
    outcomes = []
    ious = iou_matrix(pred_boxes, gt_boxes)
    gt_used = np.zeros(len(gt_boxes), dtype=bool)
    for i in np.argsort(-pred_conf):
        candidates = np.where(~gt_used & (ious[i] >= iou_thr))[0] if len(gt_boxes) else np.array([], int)
        same_class = candidates[gt_cls[candidates] == pred_cls[i]]
        pick = same_class if len(same_class) else candidates
        if len(pick):
            j = pick[np.argmax(ious[i, pick])]
            gt_used[j] = True
            kind = "correct" if gt_cls[j] == pred_cls[i] else "wrong_class"
            outcomes.append(dict(kind=kind, box=pred_boxes[i], cls=int(pred_cls[i]), gt_cls=int(gt_cls[j]),
                                 conf=float(pred_conf[i])))
        else:
            outcomes.append(dict(kind="false_pos", box=pred_boxes[i], cls=int(pred_cls[i]), gt_cls=None,
                                 conf=float(pred_conf[i])))
    for j in np.where(~gt_used)[0]:
        outcomes.append(dict(kind="missed", box=gt_boxes[j], cls=None, gt_cls=int(gt_cls[j]), conf=None))
    return outcomes

Now run the model on every test image and match. The images are passed **in batches of 16**: `ultralytics` treats a
list of images as one batch, and all 1,571 at once would need about 19 GB of GPU memory.

In [ ]:
test_images = [str((DATASET_DIR / line).resolve()) for line in (DATASET_DIR / "test.txt").read_text().split()]

per_image = []
for start in range(0, len(test_images), config.BATCH):
    for r in model.predict(source=test_images[start:start + config.BATCH], imgsz=config.IMG_SIZE,
                           conf=CONF_THRESHOLD, max_det=MAX_DET, verbose=False):
        img = Path(r.path)
        W, H = Image.open(img).size
        gt_boxes, gt_cls = load_gt(DATASET_DIR / "labels" / (img.stem + ".txt"), W, H)
        outs = match_image(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int),
                           r.boxes.conf.cpu().numpy(), gt_boxes, gt_cls)
        per_image.append(dict(image=img, n_gt=len(gt_boxes), outcomes=outs,
                              n={k: sum(o["kind"] == k for o in outs) for k in OUTCOMES}))
print(f"Matched {len(per_image):,} test images")

In [ ]:
counts = defaultdict(Counter)
confusions = Counter()
for p in per_image:
    for o in p["outcomes"]:
        cls = o["gt_cls"] if o["kind"] in ("missed", "wrong_class") else o["cls"]
        counts[o["kind"]][cls] += 1
        if o["kind"] == "wrong_class":
            confusions[(o["gt_cls"], o["cls"])] += 1

rows = []
for i, name in enumerate(CLASS_NAMES):
    gt = counts["correct"][i] + counts["wrong_class"][i] + counts["missed"][i]
    rows.append({"class": name, "ground truth": gt, "correct": counts["correct"][i],
                 "wrong class": counts["wrong_class"][i], "missed": counts["missed"][i],
                 "false positives": counts["false_pos"][i], "recall": round(counts["correct"][i] / gt, 3)})
totals = {k: sum(r[k] for r in rows) for k in ("ground truth", "correct", "wrong class", "missed", "false positives")}
rows.append({"class": "total", **totals, "recall": round(totals["correct"] / totals["ground truth"], 3)})
errors = pd.DataFrame(rows).set_index("class")
display(errors)

total = errors.loc["total"]
print(f"Missed: {100 * total['missed'] / total['ground truth']:.1f}% of all objects, "
      f"wrong class: {100 * total['wrong class'] / total['ground truth']:.1f}%")
print(f"Share of all misses that are biodegradable: {100 * errors.loc['biodegradable', 'missed'] / total['missed']:.0f}%")

In [ ]:
order = errors.drop("total").sort_values("recall", ascending=False).index
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={"width_ratios": [3, 2]})

left = np.zeros(len(order))
for kind, col, label in [("correct", AQUA, "correct"), ("wrong class", YELLOW, "wrong class"), ("missed", BLUE, "missed")]:
    vals = (100 * errors.loc[order, kind] / errors.loc[order, "ground truth"]).to_numpy(dtype=float)
    axes[0].barh(order, vals, left=left, color=col, label=label, edgecolor="white")
    for j, (l, v) in enumerate(zip(left, vals)):
        if v > 6:
            axes[0].text(l + v / 2, j, f"{v:.0f}%", ha="center", va="center", fontsize=9,
                         color=TEXT if kind == "wrong class" else "white")
    left += vals
axes[0].invert_yaxis()
axes[0].set_xlim(0, 100)
axes[0].set_title("What happens to the test objects of each class")
axes[0].set_xlabel("% of ground-truth objects")
axes[0].legend(frameon=False, ncol=3, loc="lower right")

top = confusions.most_common(8)
axes[1].barh([f"{CLASS_NAMES[t]} → {CLASS_NAMES[p]}" for (t, p), _ in top], [n for _, n in top], color=BLUE)
axes[1].invert_yaxis()
axes[1].set_title("Most common confusions (true → predicted)")
axes[1].xaxis.grid(True, color=GRID)
plt.tight_layout()
plt.show()

**Interpretation (our run).**

* **Missed objects are by far the biggest error type**: about **42%** of all test objects are not found, while only
  about **2%** get the wrong class. Improving recall matters most.
* **About two thirds of all misses are biodegradable**, almost half of its objects: tiny items in crowded piles
  (part 3.3). Biodegradable also has the most false positives, mostly in the same piles, where the labels group
  items inconsistently, so some "false positives" are real items the annotators did not box separately.
* **Confusions happen between look-alike materials**: plastic → metal and plastic → glass (shiny, transparent
  surfaces), paper ↔ cardboard (same fibre and colour), glass → metal (lids, as predicted in part 3.4).

---
## 9. Example detections

The assignment asks for **2–3 examples per case** with the boxes drawn on test images. Colours:
**green = correct**, **yellow = wrong class** (with the true class in brackets), **red = false positive**,
**blue = missed** (the ground-truth box nobody found).

We pick images with at most 15 objects, so the individual boxes stay visible, and use a fixed seed so the same
examples come back on every run.

In [ ]:
LABELS = {"correct": lambda o: f"{CLASS_NAMES[o['cls']]} {o['conf']:.2f}",
          "wrong_class": lambda o: f"{CLASS_NAMES[o['cls']]} (is {CLASS_NAMES[o['gt_cls']]})",
          "false_pos": lambda o: f"FP {CLASS_NAMES[o['cls']]} {o['conf']:.2f}",
          "missed": lambda o: f"MISSED {CLASS_NAMES[o['gt_cls']]}"}


def draw(image_path, outcomes):
    # Draw every outcome box on the image, coloured by outcome type
    im = Image.open(image_path).convert("RGB")
    d = ImageDraw.Draw(im)
    font = ImageFont.load_default()
    for o in outcomes:
        b = [float(v) for v in o["box"]]
        d.rectangle(b, outline=OUTCOMES[o["kind"]], width=3 if o["kind"] != "correct" else 2)
        text = LABELS[o["kind"]](o)
        d.rectangle([b[0], b[1], b[0] + d.textlength(text, font=font) + 4, b[1] + 11], fill=OUTCOMES[o["kind"]])
        d.text((b[0] + 2, b[1]), text, fill=TEXT if o["kind"] == "wrong_class" else "white", font=font)
    return im


def show_examples(kind, n=3, max_gt=15, seed=0):
    # Show n test images that contain at least one outcome of this kind
    pool = [p for p in per_image if p["n"][kind] > 0 and p["n_gt"] <= max_gt]
    picks = np.random.default_rng(seed).choice(len(pool), min(n, len(pool)), replace=False)
    fig, axes = plt.subplots(1, len(picks), figsize=(5.5 * len(picks), 5.8))
    for ax, k in zip(np.atleast_1d(axes), picks):
        p = pool[k]
        ax.imshow(draw(p["image"], p["outcomes"]))
        ax.axis("off")
        ax.set_title(Path(p["image"]).name.split("_jp")[0] + "   " +
                     ", ".join(f"{key.replace('_', ' ')} {v}" for key, v in p["n"].items() if v), fontsize=9)
    plt.suptitle(kind.replace("_", " ").replace("pos", "positive").capitalize(), x=0.01, ha="left", fontweight="bold")
    plt.tight_layout()
    EVAL_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(EVAL_DIR / f"examples_{kind}.png", dpi=120)
    plt.show()

### 9.1 Correct detections

In [ ]:
show_examples("correct")

Clear, well-separated objects of typical size are found with tight boxes and high confidence. Look for single
bottles, cans and boxes on clean backgrounds.

### 9.2 Missed detections

In [ ]:
show_examples("missed")

Typical reasons, visible in the examples: very small items, items in piles that overlap, items cut off at the image
border or blending into the background, and groups that the model boxes as one object while the label splits them
(one correct box plus several "missed" ones).

### 9.3 Incorrect detections: wrong class

In [ ]:
show_examples("wrong_class")

The box is in the right place but the material is wrong, mostly between look-alike classes: shiny plastic read as
metal or glass, paper read as cardboard, lids on jars.

### 9.4 Incorrect detections: false positives

In [ ]:
show_examples("false_pos")

Boxes where nothing is labelled. Some are real model errors (background texture, reflections); some are real items
that the annotators did not box separately, i.e. gaps in the labels rather than in the model.

*The example images are also saved to `results/evaluation/` for the presentation.*

---
## 10. Limitations and possible improvements

### Why the errors happen
| Cause | Evidence | Effect |
|---|---|---|
| **Tiny, crowded objects** | Half of the biodegradable boxes are under 1% of the image (3.3); up to 358 per image (3.2) | Most misses; lowest AP@[.50:.95]; NMS can merge neighbours |
| **Inconsistent labels in piles** | Some piles boxed item by item, others as one group (3.5) | Misses and false positives that are partly label artefacts |
| **Look-alike materials** | Confusion pairs in part 8 | Plastic ↔ metal / glass, paper ↔ cardboard |
| **Object imbalance** | 61% of boxes are biodegradable (3.1) | Fewer examples for rare classes such as plastic |
| **Domain** | Web photos, not conveyor-belt footage (3.5) | Real deployment images may look different |

### What would help
| Improvement | Targets | Cost |
|---|---|---|
| Larger input (832 px) or image tiling | Tiny objects | About 1.7× training time |
| Bigger model (`yolov5mu.pt`) | All classes, look-alike materials | About 2× training time |
| More epochs (80+) | mAP was still rising at epoch 50 | Longer run |
| Consistent relabelling of piles | Misses and false positives in crowds | Annotation work |
| More plastic and paper images | Rare classes, confusions | Data collection |
| Images from the real conveyor belt | Domain shift | Data collection |

Lowering the confidence threshold would also find more objects, but at the price of more false positives. For a
sorting plant, a missed item goes to the wrong bin by default, so **recall is the most important thing to improve**.

---
## Summary

* **Data:** 10,464 images, 74,073 boxes, 6 classes; balanced images but strongly unbalanced, often tiny objects;
  17 broken boxes removed.
* **Pipeline:** stratified 70/15/15 split, 640 px, default augmentation, COCO-pretrained YOLOv5s, early stopping;
  every choice follows from the analysis.
* **Result on unseen test images:** mAP@0.5 = **0.645**, mAP@[.50:.95] = **0.444**, precision 0.70, recall 0.57,
  at about 10 ms per image.
* **Main weakness:** missed objects (42%), two thirds of them tiny biodegradable items; wrong classes are rare (2%)
  and happen between look-alike materials.
* **Next step:** larger input or a bigger model to raise recall on small objects, and cleaner labels for crowded
  piles.